# 09 — Orthogonality, Projections & Gram-Schmidt
## Perpendicular Vectors and Clean Coordinate Systems

> **Prerequisites:** Notebooks 01–08  
> **Difficulty:** ⭐⭐⭐☆☆ Intermediate  
> **Running example:** Feature decorrelation, least squares geometry

---

## Why does this matter?

Imagine measuring a room's dimensions. You measure length along one wall and width along a **perpendicular** wall. You get two independent measurements.

If instead you measured at two angles that aren't 90° apart — say 70° and 70° — your measurements would overlap in information. One measurement would partially capture what the other already captured.

**Orthogonal (perpendicular) directions give you maximum independence and minimum redundancy.** This is why orthogonal bases are so useful in ML:

| Application | How orthogonality helps |
|---|---|
| **PCA** | Principal components are orthogonal → uncorrelated features |
| **Least squares** | The residual is orthogonal to the predicted values |
| **QR decomposition** | Stable way to solve linear systems |
| **Stable training** | Orthogonal weight initialization prevents gradient explosion |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## 1. Orthogonality — Perpendicular Vectors

### Plain English
Two vectors are **orthogonal** if they are at **90°** to each other.

The dot product test:
> **u · v = 0  ↔  u and v are orthogonal**

(From the geometric formula: u · v = ‖u‖ ‖v‖ cos(θ). When θ=90°, cos(90°)=0.)

### Orthonormal vectors
**Orthonormal** = orthogonal **AND** both have unit length (length = 1).

The standard basis vectors [1,0] and [0,1] are orthonormal!

### Why orthogonal = independent?
Orthogonal vectors have **zero correlation** — they capture completely different aspects of the data. This is the most "independent" two vectors can be.

In [ ]:
# ─────────────────────────────────────────────────────────
# ORTHOGONALITY CHECK
# ─────────────────────────────────────────────────────────

def is_orthogonal(u, v, tol=1e-9):
    """Two vectors are orthogonal if their dot product is zero."""
    return abs(np.dot(u, v)) < tol

def is_unit_length(v, tol=1e-9):
    """Check if a vector has length 1."""
    return abs(np.linalg.norm(v) - 1.0) < tol

print("=== Orthogonality tests ===")
print()

# Standard basis — orthonormal
e1 = np.array([1., 0., 0.])
e2 = np.array([0., 1., 0.])
e3 = np.array([0., 0., 1.])
print("Standard basis vectors:")
for i, (a, b, la, lb) in enumerate([(e1,e2,'e1','e2'), (e1,e3,'e1','e3'), (e2,e3,'e2','e3')]):
    print(f"  {la}·{lb} = {np.dot(a,b)}, orthogonal: {is_orthogonal(a,b)}")
print(f"  All unit length: {all(is_unit_length(v) for v in [e1,e2,e3])}")
print()

# Non-orthogonal vectors
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])
print(f"u = {u}, v = {v}")
print(f"u·v = {np.dot(u,v)}  → orthogonal: {is_orthogonal(u,v)}")

print()
print("=== Making vectors orthogonal ===")
# Given u = [1, 2], find a vector perpendicular to it in 2D
u2d = np.array([3., 2.])
v_perp = np.array([-u2d[1], u2d[0]])   # rotate 90°: [a,b] → [-b,a]
print(f"u = {u2d}")
print(f"v_perp = {v_perp}  (rotated 90°)")
print(f"u · v_perp = {np.dot(u2d, v_perp)}  (zero ✓)")

---
## 2. Orthogonal Matrices — Transformations That Preserve Shape

### What is it?
A square matrix Q is **orthogonal** if its columns are **orthonormal**:
> **QᵀQ = QQᵀ = I** → **Qᵀ = Q⁻¹**

### Key property: orthogonal matrices preserve everything
- **Preserve lengths**: ‖Qv‖ = ‖v‖ for all v
- **Preserve angles**: (Qa)·(Qb) = a·b
- **det(Q) = ±1** (no scaling, only rotation/reflection)

### Why in ML?
- Rotation matrices are orthogonal
- U and V from SVD are orthogonal — this is why SVD is numerically stable
- Orthogonal weight initialization helps neural networks train better

In [ ]:
# ─────────────────────────────────────────────────────────
# ORTHOGONAL MATRICES
# ─────────────────────────────────────────────────────────

print("=== Orthogonal matrix properties ===")
print()

# Get an orthogonal matrix via QR decomposition of a random matrix
np.random.seed(0)
random_matrix = np.random.randn(4, 4)
Q, _ = np.linalg.qr(random_matrix)

print(f"Random orthogonal matrix Q (4×4):")
print(Q.round(4))
print()
print(f"QᵀQ = I? {np.allclose(Q.T @ Q, np.eye(4))}  ← columns are orthonormal")
print(f"QQᵀ = I? {np.allclose(Q @ Q.T, np.eye(4))}")
print(f"Qᵀ = Q⁻¹? {np.allclose(Q.T, np.linalg.inv(Q))}")  
print(f"det(Q) = {np.linalg.det(Q):.6f}  (should be ±1)")
print()

# Show it preserves lengths and angles
v1 = np.array([1., 2., -1., 3.])
v2 = np.array([0., 1.,  2., -1.])
Qv1 = Q @ v1; Qv2 = Q @ v2

print("Q preserves lengths:")
print(f"  ‖v1‖ = {np.linalg.norm(v1):.6f},  ‖Qv1‖ = {np.linalg.norm(Qv1):.6f}")
print()
print("Q preserves angles (dot products):")
print(f"  v1·v2  = {np.dot(v1,v2):.6f}")
print(f"  Qv1·Qv2 = {np.dot(Qv1,Qv2):.6f}")

---
## 3. Projections — Finding the Closest Point

### Plain English
**Projection** of vector b onto vector a = the point on the line through a that is **closest to b**.

Think of it as: drop a perpendicular from b down to the line of a. Where it lands is the projection.

### Formula
> **proj_a(b) = (a·b / a·a) × a = (a·b / ‖a‖²) × a**

### The residual
The **residual** = b - proj_a(b) = the part of b **perpendicular** to a.

### Why in ML?
- **Least squares**: the predicted values ŷ are the projection of y onto the column space of X. The residuals (y - ŷ) are orthogonal to ŷ.
- **PCA**: projecting data onto principal components
- **Gram-Schmidt** (next section): builds orthogonal bases using projections

In [ ]:
# ─────────────────────────────────────────────────────────
# VECTOR PROJECTION
# ─────────────────────────────────────────────────────────

print("=== Projecting vector b onto vector a ===")
print()

a = np.array([4., 1.])
b = np.array([2., 4.])

# Compute projection
scalar_proj = np.dot(a, b) / np.dot(a, a)   # scalar: how far along a
vector_proj = scalar_proj * a                # vector: the projection itself
residual    = b - vector_proj                # perpendicular component

print(f"a = {a}  (the direction to project onto)")
print(f"b = {b}  (the vector we're projecting)")
print(f"")
print(f"scalar projection = a·b / ‖a‖² = {np.dot(a,b)} / {np.dot(a,a)} = {scalar_proj:.4f}")
print(f"vector projection  = {scalar_proj:.4f} × {a} = {vector_proj.round(4)}")
print(f"residual (perp)    = b - proj = {residual.round(4)}")
print(f"")
print(f"proj · residual = {np.dot(vector_proj, residual):.6e}  (near zero = perpendicular ✓)")

# Visualize
fig, ax = plt.subplots(figsize=(7, 6))
ax.annotate('', xy=a, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.text(a[0]+0.1, a[1]+0.1, 'a', color='blue', fontsize=14)
ax.annotate('', xy=b, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.text(b[0]+0.1, b[1]+0.1, 'b', color='green', fontsize=14)
ax.annotate('', xy=vector_proj, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(vector_proj[0]+0.1, vector_proj[1]-0.3, 'proj_a(b)', color='red', fontsize=12)
# Draw the perpendicular line
ax.plot([vector_proj[0], b[0]], [vector_proj[1], b[1]], 'k--', lw=1.5, label='residual (⊥ to a)')
# Right angle marker
dx = 0.2; dy = 0.2
ax.plot([vector_proj[0]+dy*(b[1]-vector_proj[1])/np.linalg.norm(b-vector_proj),
         vector_proj[0]+dy*(b[1]-vector_proj[1])/np.linalg.norm(b-vector_proj) + dx*(b[0]-vector_proj[0])/np.linalg.norm(b-vector_proj)],
        [vector_proj[1]-dy*(b[0]-vector_proj[0])/np.linalg.norm(b-vector_proj),
         vector_proj[1]-dy*(b[0]-vector_proj[0])/np.linalg.norm(b-vector_proj) + dy*(b[1]-vector_proj[1])/np.linalg.norm(b-vector_proj)],
        'k-', lw=1)
ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_title('Projection: closest point on a\'s line to b', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Gram-Schmidt — Building an Orthogonal Basis

### The problem
You have a set of vectors that span a space, but they're not orthogonal (they're pointing in similar directions, creating redundancy).

You want to find a **new set of orthonormal vectors** that spans the **same space**.

### The algorithm (step by step)

Given vectors v₁, v₂, v₃, ...:

**Step 1:** Take v₁ as-is → normalize to get **e₁ = v₁ / ‖v₁‖**

**Step 2:** Remove the e₁ component from v₂:
> u₂ = v₂ - (v₂ · e₁) × e₁
Then normalize: **e₂ = u₂ / ‖u₂‖**

**Step 3:** Remove both e₁ and e₂ components from v₃:
> u₃ = v₃ - (v₃ · e₁)e₁ - (v₃ · e₂)e₂
Then normalize: **e₃ = u₃ / ‖u₃‖**

And so on. Each step **projects away** what was already captured.

In [ ]:
# ─────────────────────────────────────────────────────────
# GRAM-SCHMIDT from scratch
# ─────────────────────────────────────────────────────────

def gram_schmidt(vectors):
    """
    Input:  list of linearly independent vectors (as numpy arrays)
    Output: list of orthonormal vectors spanning the same space
    """
    orthonormal = []  # will store our orthonormal vectors as we build them
    
    for v in vectors:
        v = v.copy().astype(float)
        
        # Remove the projection onto each previously found orthonormal vector
        for e in orthonormal:
            projection_amount = np.dot(e, v)  # how much of v lies along e
            v = v - projection_amount * e     # subtract that component
        
        # Normalize what remains to unit length
        length = np.linalg.norm(v)
        if length < 1e-10:
            raise ValueError("Vectors are linearly dependent — cannot make orthonormal basis")
        v = v / length
        orthonormal.append(v)
    
    return orthonormal

print("=== Gram-Schmidt: step-by-step ===")
print()

# Three vectors in ℝ³ that aren't orthogonal
v1 = np.array([1., 1., 0.])
v2 = np.array([1., 0., 1.])
v3 = np.array([0., 1., 1.])
vectors = [v1, v2, v3]

print("Input vectors (not orthogonal):")
for i, v in enumerate(vectors, 1):
    print(f"  v{i} = {v}")
print()
print(f"v1·v2 = {np.dot(v1,v2)}  (not zero, so not orthogonal)")
print(f"v1·v3 = {np.dot(v1,v3)}  (not zero)")
print()

# Run Gram-Schmidt
orthonormal = gram_schmidt(vectors)

print("Output: orthonormal basis:")
for i, e in enumerate(orthonormal, 1):
    print(f"  e{i} = {e.round(4)}")
print()
print("Verify orthonormality:")
for i in range(len(orthonormal)):
    for j in range(i, len(orthonormal)):
        dot = np.dot(orthonormal[i], orthonormal[j])
        if i == j:
            print(f"  e{i+1}·e{i+1} = {dot:.4f}  (unit length? {abs(dot-1)<1e-9})")
        else:
            print(f"  e{i+1}·e{j+1} = {dot:.6e}  (orthogonal? {abs(dot)<1e-9})")

In [ ]:
# ─────────────────────────────────────────────────────────
# QR DECOMPOSITION: production-grade Gram-Schmidt
# ─────────────────────────────────────────────────────────
# np.linalg.qr is the stable, optimized version of Gram-Schmidt

print("=== QR Decomposition ===")
print("Every matrix A can be factored as A = Q @ R")
print("  Q: orthogonal matrix (orthonormal columns)")
print("  R: upper triangular matrix")
print()

np.random.seed(42)
A = np.random.randn(5, 3)   # 5×3 matrix (overdetermined)

Q, R = np.linalg.qr(A)

print(f"A shape: {A.shape}")
print(f"Q shape: {Q.shape}  (orthonormal columns)")
print(f"R shape: {R.shape}  (upper triangular)")
print()
print(f"QᵀQ = I? {np.allclose(Q.T @ Q, np.eye(3))}")
print(f"A = Q@R? {np.allclose(A, Q @ R)}")
print()
print("R (upper triangular):")
print(R.round(4))
print()
print("=== Solving Ax ≈ b more stably with QR ===")
b = np.random.randn(5)

# QR approach: solve Q @ R @ x = b → R @ x = Qᵀ @ b
x_qr = np.linalg.solve(R, Q.T @ b)
x_lstsq, *_ = np.linalg.lstsq(A, b, rcond=None)
print(f"QR solution: {x_qr.round(4)}")
print(f"lstsq:       {x_lstsq.round(4)}")
print(f"Match: {np.allclose(x_qr, x_lstsq)}")

---
## Summary

| Concept | What it means | NumPy |
|---|---|---|
| Orthogonal vectors | u·v = 0 (90° angle) | `np.dot(u,v) == 0` |
| Orthonormal | orthogonal + unit length | orthogonal AND `np.linalg.norm(v) == 1` |
| Orthogonal matrix Q | QᵀQ = I, det = ±1 | `np.linalg.qr(A)[0]` |
| Projection | Closest point on a line to a vector | `(np.dot(a,b)/np.dot(a,a)) * a` |
| Gram-Schmidt | Convert any basis to orthonormal | manual (shown above) |
| QR decomposition | A = QR, Q orthogonal, R triangular | `np.linalg.qr(A)` |

**Next: Notebook 10 — Special Matrices** (identity, diagonal, symmetric, positive definite)